<a href="https://colab.research.google.com/github/samuelrossiello/data-analytics-portfolio/blob/main/bootcamp/week5_sql_python_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Exercise 1

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retail.db")
print("Connected!")

Connected!


### Exercise 2

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "customer_name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "region": ["North", "South", "East", "West", "North"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "customer_id": [1, 2, 1, 3, 5, 6],
    "product": ["T-Shirt", "Jeans", "Sneakers", "Jacket", "Hat", "Boots"],
    "revenue": [59.97, 99.98, 74.99, 44.43, 29.99, 89.99]
})

# Load DataFrames into SQLite as tables
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

print("Tables created!")

Tables created!


### Exercise 3

In [ ]:
result = pd.read_sql("SELECT * FROM customers", conn)

print(result)

   customer_id customer_name region
0            1         Alice  North
1            2           Bob  South
2            3         Carol   East
3            4          Dave   West
4            5           Eve  North


In [ ]:
result=pd.read_sql("SELECT * FROM orders WHERE revenue > 50", conn)

print(result)

   order_id  customer_id   product  revenue
0       101            1   T-Shirt    59.97
1       102            2     Jeans    99.98
2       103            1  Sneakers    74.99
3       106            6     Boots    89.99


In [ ]:
result = pd.read_sql("""
SELECT
    customer_id,
    COUNT(*) as order_count
    FROM orders
    GROUP BY customer_id
    """, conn
)

print(result)

   customer_id  order_count
0            1            2
1            2            1
2            3            1
3            5            1
4            6            1


### Exercise 4

In [ ]:
query = """
  SELECT
    c.customer_name,
    c.region,
    o.product,
    o.revenue
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.revenue > 40
"""

df = pd.read_sql(query, conn)

df["revenue_rank"] = df["revenue"].rank(ascending=False)
df["running_total"] = df["revenue"].cumsum()

print(df)

  customer_name region   product  revenue  revenue_rank  running_total
0         Alice  North   T-Shirt    59.97           4.0          59.97
1           Bob  South     Jeans    99.98           1.0         159.95
2         Alice  North  Sneakers    74.99           3.0         234.94
3         Carol   East    Jacket    44.43           5.0         279.37
4          None   None     Boots    89.99           2.0         369.36


### Exercise 5

In [ ]:
df.to_sql("order_analysis", conn, if_exists="replace", index=False)

verification = pd.read_sql("SELECT * FROM order_analysis", conn)

print(verification)

  customer_name region   product  revenue  revenue_rank  running_total
0         Alice  North   T-Shirt    59.97           4.0          59.97
1           Bob  South     Jeans    99.98           1.0         159.95
2         Alice  North  Sneakers    74.99           3.0         234.94
3         Carol   East    Jacket    44.43           5.0         279.37
4          None   None     Boots    89.99           2.0         369.36


### Practice Project

In [ ]:
query = """
  SELECT
    c.customer_name,
    c.region,
    SUM(o.revenue) AS total_revenue
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_name, c.region
    HAVING COUNT(order_id) > 1
"""

df = pd.read_sql(query, conn)

print(df)

  customer_name region  total_revenue
0         Alice  North         134.96


In [ ]:
def revenue_tier(total_revenue):
    if total_revenue > 100:
        return "High"
    else:
        return "Standard"

df["revenue_tier"] = df["total_revenue"].apply(revenue_tier)

print(df)

  customer_name region  total_revenue revenue_tier
0         Alice  North         134.96         High


In [ ]:
df.to_sql("customer_summary", conn, if_exists="replace", index=False)

verification = pd.read_sql("SELECT * FROM customer_summary", conn)

print(verification)

  customer_name region  total_revenue revenue_tier
0         Alice  North         134.96         High
